In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Perceptron
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

digits = load_digits()
X, y = digits.data, digits.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [3]:
perceptron = Perceptron(max_iter=1000, random_state=42)
perceptron.fit(X_train, y_train)

y_pred_perc = perceptron.predict(X_test)
perc_accuracy = accuracy_score(y_test, y_pred_perc)

print(f"Perceptron Accuracy: {perc_accuracy:.4f}")


Perceptron Accuracy: 0.9500


In [4]:
mlp = MLPClassifier(hidden_layer_sizes=(50, 50), max_iter=1000, random_state=42)
mlp.fit(X_train, y_train)

y_pred_mlp = mlp.predict(X_test)
mlp_accuracy = accuracy_score(y_test, y_pred_mlp)

print(f"MLP Accuracy: {mlp_accuracy:.4f}")

MLP Accuracy: 0.9722


In [11]:
conf_matrix = confusion_matrix(y_test, y_pred_mlp)
class_report = classification_report(y_test, y_pred_mlp)


print("Confusion Matrix:")
display(conf_matrix)
print("\nClassification Report:")
print(class_report)

Confusion Matrix:


array([[33,  0,  0,  0,  0,  0,  0,  0,  0,  0],
       [ 0, 28,  0,  0,  0,  0,  0,  0,  0,  0],
       [ 0,  0, 31,  2,  0,  0,  0,  0,  0,  0],
       [ 0,  0,  0, 34,  0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0, 46,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0, 45,  1,  0,  0,  1],
       [ 0,  0,  0,  0,  0,  0, 35,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0,  0, 33,  0,  1],
       [ 0,  2,  0,  0,  0,  1,  0,  0, 27,  0],
       [ 0,  0,  0,  1,  0,  0,  0,  0,  1, 38]])


Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        33
           1       0.93      1.00      0.97        28
           2       1.00      0.94      0.97        33
           3       0.92      1.00      0.96        34
           4       1.00      1.00      1.00        46
           5       0.98      0.96      0.97        47
           6       0.97      1.00      0.99        35
           7       1.00      0.97      0.99        34
           8       0.96      0.90      0.93        30
           9       0.95      0.95      0.95        40

    accuracy                           0.97       360
   macro avg       0.97      0.97      0.97       360
weighted avg       0.97      0.97      0.97       360



### Mathematical Breakdown: Why the MLP Succeeds

Consider a specific test sample $\mathbf{x} \in \mathbb{R}^{64}$ representing one of the historical seal digits.

#### 1. The Limitation of the Single-Layer Perceptron

A standard Perceptron computes a linear combination of the inputs for each class $c$:


$$\mathbf{z}_c = \mathbf{w}_c^T \mathbf{x} + b_c$$


It can only partition the 64-dimensional feature space using flat hyperplanes. If two different digits share overlapping pixel densities in certain linear projections, a linear hyperplane cannot separate them without misclassifying others.

#### 2. The Power of Hidden Layers and Non-Linearity in the MLP

When we introduce hidden layers with the Rectified Linear Unit ($\text{ReLU}$) activation function, the network transforms the input space through successive non-linear mappings:

* **First Hidden Layer ($\mathbf{h}_1$):**

$$\mathbf{h}_1 = \max(0, \mathbf{W}_1 \mathbf{x} + \mathbf{b}_1)$$



Here, $\mathbf{W}_1$ is a $50 \times 64$ weight matrix. The linear projection $\mathbf{W}_1 \mathbf{x} + \mathbf{b}_1$ defines 50 hyperplanes in the 64-dimensional space. The $\max(0, \cdot)$ function (ReLU) acts as a non-linear gating mechanism, zeroing out negative values and passing positive responses.
* **Second Hidden Layer ($\mathbf{h}_2$):**

$$\mathbf{h}_2 = \max(0, \mathbf{W}_2 \mathbf{h}_1 + \mathbf{b}_2)$$



This second $50 \times 50$ layer takes the features extracted by the first layer and folds, bends, and warps the feature space further.
* **Output Layer:**

$$\hat{\mathbf{y}} = \text{softmax}(\mathbf{W}_{\text{out}} \mathbf{h}_2 + \mathbf{b}_{\text{out}})$$



#### Why This Solves the Problem

By chaining these non-linear transformations, the MLP constructs complex, piecewise-linear decision boundaries. Instead of drawing a single flat cut across the 64-dimensional space, the MLP essentially reshapes the feature space so that ambiguous digit variations (like a messy handwritten '4' that looks slightly like a '9') are pulled apart into linearly separable regions before final classification. and that's how we prove that MLP is better than a single Perceptron and the result supports it